# pycograd — RWKV, a recurrent net in the ambient-weights DSL

[RWKV](https://github.com/codekansas/rwkv) is a sequence model with a twist: it
*trains* like a transformer — one parallel pass over the whole sequence — but
*runs* like an RNN, carrying a fixed-size state forward one token at a time
(O(1) work and memory per token, no growing attention cache).

This notebook builds a tiny **character-level RWKV language model** with pycograd's
ambient-weights DSL: declare the parameters once with `params{...}`, write the
forward once, and the *same* code trains on the numpy tape, batches under `vmap`,
and compiles to **torch / jax / tf**. At the end we sample text using the O(1)
recurrent form.

The recurrence is built from ordinary numpy (`exp`, `maximum`, matmul, indexing),
so every transform composes with it for free. The reusable pieces — `token_shift`,
the numerically-stable `wkv` kernel, the mixing blocks — ship in
`pycograd.examples.models`.

In [1]:
%load_ext pycograd

## A tiny corpus

Character-level: the model predicts the next character from the prefix so far.
The corpus is deliberately small so a tiny model can memorize it quickly.

In [2]:
import numpy as np
from pycograd import cross_entropy, relu, sigmoid, vmap
from pycograd.examples import train, accuracy
from pycograd.examples.models import token_shift, wkv, layer_norm

text = "pycograd makes rwkv tiny. "
vocab = sorted(set(text))
stoi = {c: i for i, c in enumerate(vocab)}
ids = np.array([stoi[c] for c in text])
onehot = np.eye(len(vocab))[ids]            # (T, V) one-hot rows
V = len(vocab)
print("text  :", repr(text))
print("vocab :", "".join(vocab), " (V=%d, T=%d)" % (V, len(ids)))

# inputs predict the *next* char: feed prefix 0..T-2, target chars 1..T-1
X_oh, Y_oh, Y_ids = onehot[:-1], onehot[1:], ids[1:]

text  : 'pycograd makes rwkv tiny. '
vocab :  .acdegikmnoprstvwy  (V=19, T=26)


## The model — one RWKV block, written once

An RWKV block is two residual sub-layers, each fed a **token-shift** mix of the
current and previous timestep (`token_shift` prepends a zero row and drops the
last):

- **Time-mixing (the "attention"):** project the mix into key/value/receptance,
  run the `wkv` kernel — a softmax-weighted running average of values that decays
  by `exp(-w)` per step and gives the current step a `u` bonus — then gate by
  `sigmoid(r)` and project out. `wkv` carries its running numerator/denominator in
  a max-normalized form, so `exp` never overflows however long the sequence.
- **Channel-mixing (the FFN):** a squared-ReLU hidden layer gated by `sigmoid(r)`.

The embedding is `onehot @ emb` — an embedding lookup written as a matmul, which
keeps the whole net free of fancy indexing so it lowers to every backend.

We write the forward `rwkv` once, as a top-level function. It refers to the
weights by **bare name** (`emb`, `Wk`, …); `with params{...} as weights:` injects a
late-bound proxy for each into the notebook namespace, so the *same* `rwkv` is
bound to tape nodes by `weights.grad`, to a batch by `vmap`, and to framework
tensors by the compiler — no rewrite per use.

In [3]:
def rwkv(oh):                                      # oh: (T, V) one-hot rows
    x = oh @ emb                                   # (T, D) embedding (a matmul)
    x = layer_norm(x, ln0_g, ln0_b)

    xn = layer_norm(x, ln1_g, ln1_b)               # --- time mixing (attention) ---
    sx = token_shift(xn)
    k = (xn*mix_k + sx*(1 - mix_k)) @ Wk
    v = (xn*mix_v + sx*(1 - mix_v)) @ Wv
    r = sigmoid((xn*mix_r + sx*(1 - mix_r)) @ Wr)
    x = x + (wkv(np.exp(decay), bonus, k, v) * r) @ Wo

    xn = layer_norm(x, ln2_g, ln2_b)               # --- channel mixing (FFN) ---
    sx = token_shift(xn)
    h = relu((xn*fmix_k + sx*(1 - fmix_k)) @ fWk)
    x = x + sigmoid((xn*fmix_r + sx*(1 - fmix_r)) @ fWr) * ((h*h) @ fWv)

    x = layer_norm(x, lnf_g, lnf_b)                # --- head ---
    return x @ head_w + head_b

objective = |> X_oh |> rwkv($) |> cross_entropy($, Y_oh)

Now declare the weights with `params{...}` and train. The block names below are
exactly the bare names `rwkv` refers to; `train(...)` (from `pycograd.examples`)
runs `weights.grad(objective)` and `weights.step` each step, binding the names to
the numpy tape and updating them in place.

In [4]:
rng = np.random.default_rng(0)
D = 24                                       # model width
s = 0.1

with params{
    emb       = s * rng.standard_normal((V, D))
    ln0_g     = np.ones(D);  ln0_b = np.zeros(D)
    # time-mixing (attention)
    ln1_g     = np.ones(D);  ln1_b = np.zeros(D)
    decay     = np.zeros(D)                   # w = exp(decay) is the per-channel decay
    bonus     = np.zeros(D)                   # u, the current-step bonus
    mix_k     = rng.random(D); mix_v = rng.random(D); mix_r = rng.random(D)
    Wk = s*rng.standard_normal((D, D)); Wv = s*rng.standard_normal((D, D))
    Wr = s*rng.standard_normal((D, D)); Wo = s*rng.standard_normal((D, D))
    # channel-mixing (FFN)
    ln2_g     = np.ones(D);  ln2_b = np.zeros(D)
    fmix_k    = rng.random(D); fmix_r = rng.random(D)
    fWk = s*rng.standard_normal((D, 4*D)); fWr = s*rng.standard_normal((D, D))
    fWv = s*rng.standard_normal((4*D, D))
    # head
    lnf_g     = np.ones(D);  lnf_b = np.zeros(D)
    head_w = s*rng.standard_normal((D, V)); head_b = np.zeros(V)
} as weights:
    first, last = train(weights, objective, 200, 0.3)
    logits = rwkv(X_oh)

print("loss %.4f -> %.4f  (200 SGD steps)" % (first, last))
print("train accuracy:", accuracy(logits, Y_ids))

loss 3.0939 -> 0.0052  (200 SGD steps)
train accuracy: 1.0


## Batching with `vmap`

`rwkv` was written for one `(T, V)` sequence. `vmap` vectorizes it over a batch of
sequences in a single pass — no Python loop — and the result matches the loop
oracle exactly. The token-shift and the per-timestep `wkv` recurrence batch for
free, because `vmap` presents each sequence its own *logical* shape.

In [5]:
# a batch of (T, V) one-hot sequences (random tokens, just to exercise batching)
B, T = 5, len(X_oh)
batch = np.eye(V)[np.random.default_rng(1).integers(0, V, size=(B, T))]

with weights:
    batched = np.asarray(vmap(rwkv)(batch))                 # one vectorized pass
    looped  = np.stack([np.asarray(rwkv(batch[i])) for i in range(B)])  # oracle

print("vmap output shape:", batched.shape)
print("matches the python loop:", np.allclose(batched, looped, atol=1e-9))

vmap output shape: (5, 25, 19)
matches the python loop: True


## Compiling to torch / jax / tf

`weights.grad(objective, backend=...)` runs the *same* objective through a
framework's own autodiff instead of pycograd's numpy tape. The recurrence — token
shift, the WKV scan, the sigmoid gates — lowers cleanly to all three. We check the
framework gradients against the numpy tape (which is finite-difference-checked in
the test suite), so agreement validates the whole compile path.

`jit=True` traces and compiles the gradient **once** (jax.jit / tf.function /
torch make_fx), not per step.

In [6]:
with weights:
    v_np, g_np = weights.grad(objective)                    # pycograd's numpy tape
    for backend in ("torch", "jax", "tf"):
        v_be, g_be = weights.grad(objective, backend=backend, jit=True)
        worst = max(np.max(np.abs(np.asarray(g_be[k]) - np.asarray(g_np[k]))) for k in weights)
        print("%-5s  value=%.6f  max|grad - grad_numpy| = %.1e" % (backend, float(v_be), worst))
    print("numpy value=%.6f" % float(v_np))

torch  value=0.005198  max|grad - grad_numpy| = 2.3e-16


jax    value=0.005198  max|grad - grad_numpy| = 1.4e-16


tf     value=0.005198  max|grad - grad_numpy| = 9.0e-17
numpy value=0.005198


## Generating text — the O(1) recurrent form

Training used the parallel form (the whole sequence at once). For sampling we use
the **recurrent** form: carry the token-shift state plus the WKV running
`(num, den, max)` accumulators forward one character at a time. Each step is O(1)
— no re-reading the prefix — yet produces exactly the logits the parallel form
would. We read the trained weights straight out of the `ParamDict`.

In [7]:
W = {k: np.asarray(weights[k].value) for k in weights}   # trained arrays

def generate(prime, n_new, temp=0.5, seed=0):
    g = np.random.default_rng(seed)
    # per-block recurrent state: token-shift carries + WKV (alpha, beta, eps)
    att_x = np.zeros(D); alpha = np.zeros(D); beta = np.zeros(D); eps = np.zeros(D)
    ffn_x = np.zeros(D)
    out = list(prime)

    def step(token):
        nonlocal att_x, alpha, beta, eps, ffn_x
        x = layer_norm(W["emb"][token], W["ln0_g"], W["ln0_b"])
        xn = layer_norm(x, W["ln1_g"], W["ln1_b"])
        k = (xn*W["mix_k"] + att_x*(1-W["mix_k"])) @ W["Wk"]
        v = (xn*W["mix_v"] + att_x*(1-W["mix_v"])) @ W["Wv"]
        r = sigmoid((xn*W["mix_r"] + att_x*(1-W["mix_r"])) @ W["Wr"])
        w, u = np.exp(W["decay"]), W["bonus"]
        tau = np.maximum(u + k, eps)
        e1, e2 = np.exp(eps - tau), np.exp(u + k - tau)
        wkv_t = (e1*alpha + e2*v) / (e1*beta + e2)
        eps2 = np.maximum(eps - w, k)
        a1, a2 = np.exp(eps - w - eps2), np.exp(k - eps2)
        alpha, beta, eps = a1*alpha + a2*v, a1*beta + a2, eps2
        att_x = xn
        x = x + (wkv_t * r) @ W["Wo"]
        xn2 = layer_norm(x, W["ln2_g"], W["ln2_b"])
        h = relu((xn2*W["fmix_k"] + ffn_x*(1-W["fmix_k"])) @ W["fWk"])
        x = x + sigmoid((xn2*W["fmix_r"] + ffn_x*(1-W["fmix_r"])) @ W["fWr"]) * ((h*h) @ W["fWv"])
        ffn_x = xn2
        x = layer_norm(x, W["lnf_g"], W["lnf_b"])
        return x @ W["head_w"] + W["head_b"]

    logits = None
    for c in prime:                                  # warm the state on the prime
        logits = step(stoi[c])
    for _ in range(n_new):
        p = np.exp((logits - logits.max()) / temp); p /= p.sum()
        nxt = int(g.choice(V, p=p))
        out.append(vocab[nxt])
        logits = step(nxt)
    return "".join(out)

print(repr(generate("pyco", 22)))

'pycograd makes rwkv tiny. '


## The fused `d_sigmoid` primitive

The gates above use a numpy-composed `sigmoid` (so plain-array inference works).
pycograd also ships a fused **`d_sigmoid`** primitive: one tape node with the
closed-form derivative `s·(1−s)` instead of an `exp`/`reciprocal` chain. It is a
first-class primitive — it carries vjp, jvp, vmap and shape rules and lowers to
`torch.sigmoid` / `jax.nn.sigmoid` / `tf.math.sigmoid` — so you can call it
directly under any transform.

In [8]:
from pycograd import d_sigmoid, grad

x = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
(g,) = grad(lambda z: d_sigmoid(z) @ np.ones(5))(x)
s = 1.0 / (1.0 + np.exp(-x))
print("d_sigmoid'(x) :", np.round(g, 4))
print("s * (1 - s)   :", np.round(s * (1 - s), 4))

d_sigmoid'(x) : [0.105 0.235 0.25  0.235 0.105]
s * (1 - s)   : [0.105 0.235 0.25  0.235 0.105]


## More

- `pycograd_demo.ipynb` — the DSL from scratch (MLP, highway nets, attention, a Transformer block).
- `pycograd_vmap_demo.ipynb` — batching with `vmap`.
- `pycograd_compile_{torch,jax,tf}_demo.ipynb` — compiling the DSL to each framework.

The RWKV building blocks (`token_shift`, `wkv`, `rwkv_time_mixing`,
`rwkv_channel_mixing`, `rwkv_block`, and the recurrent `rwkv_step`) live in
`pycograd.examples.models`.